# Kestrel M1 · 01 Profiling

**Goal of this notebook:** read every batch file in its own format, profile it, and write down **what is wrong
before anything is fixed**. Nothing here repairs data. Each problem is logged with *how it was detected* and a
*planned decision* into a findings log, exported at the end to `output/profiling/findings.csv` (the input to
`docs/dq_findings.md`).

Readers live in `kestrel_m1/readers.py`, helpers in `kestrel_m1/profiling.py`.
`stream_events.json` and `incident_labels.csv` are Milestone 2 inputs: they appear in the byte inventory only and are
**not** read or profiled here (the labels are held-out ground truth and must never feed the pipeline).

In [1]:
import sys, pathlib, re
ROOT = pathlib.Path.cwd().parent          # notebooks/ -> M1/
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from kestrel_m1 import readers as rd
from kestrel_m1.profiling import (profile_frame, value_shapes, enum_violations, fk_orphans,
                                  numeric_ranges, interval_overlaps, host_matches, FindingsLog)

pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
F = FindingsLog()

## 1. Inventory: bytes, encoding, line endings

The pack convention is UTF-8, `\n` line endings, RFC 4180 CSV with a header. Checked at byte level before parsing.

In [2]:
inv = rd.file_inventory()
inv

,file,format,used_in,bytes,lines,crlf_lines,utf8,ends_with_newline,rows_in_footer,row_groups,compression
0,tenants.csv,csv,M1,677,7.0,0.0,True,True,NaN,NaN,NaN
1,agents.csv,csv,M1,2876,23.0,0.0,True,True,NaN,NaN,NaN
2,tools.csv,csv,M1,1265,19.0,0.0,True,True,NaN,NaN,NaN
3,model_pricing.csv,csv,M1,436,6.0,0.0,True,True,NaN,NaN,NaN
4,sandbox_policies.json,nested_json,M1,7003,354.0,0.0,True,True,NaN,NaN,NaN
5,agent_activity.parquet,parquet,M1,33272064,NaN,NaN,NaN,NaN,937752.0,1.0,ZSTD
6,network_flows.csv,csv,M1,5595344,42393.0,0.0,True,True,NaN,NaN,NaN
7,sandbox_audit.log,syslog,M1,34698060,184005.0,0.0,True,True,NaN,NaN,NaN
8,siem_alerts.json,nested_json,M1,1646681,63824.0,0.0,True,True,NaN,NaN,NaN
9,stream_events.json,json_array,M2,9779879,304198.0,0.0,True,False,NaN,NaN,NaN


In [3]:
text = inv[inv.format != "parquet"]
print("non-UTF-8 text files:", list(text.loc[~text.utf8.astype(bool), "file"]))
print("files with CRLF line endings:", list(text.loc[text.crlf_lines > 0, "file"]))

non-UTF-8 text files: []
files with CRLF line endings: []


In [4]:
tenants  = rd.read_tenants()
agents   = rd.read_agents()
tools    = rd.read_tools()
pricing  = rd.read_model_pricing()
policies = rd.read_sandbox_policies()
flows    = rd.read_network_flows()
siem_x   = rd.read_siem_alerts(); siem = siem_x.alerts
audit_x  = rd.read_sandbox_audit(); audit = audit_x.parsed
act      = rd.read_activity()

documented = {"tenants.csv": 6, "agents.csv": 22, "tools.csv": 18, "model_pricing.csv": 5,
              "sandbox_policies.json": 10, "agent_activity.parquet": 940_000, "network_flows.csv": 42_000,
              "sandbox_audit.log": 184_000, "siem_alerts.json": 3_800}
loaded = {"tenants.csv": len(tenants), "agents.csv": len(agents), "tools.csv": len(tools),
          "model_pricing.csv": len(pricing), "sandbox_policies.json": len(policies),
          "agent_activity.parquet": len(act), "network_flows.csv": len(flows),
          "sandbox_audit.log": len(audit) + len(audit_x.unparsed), "siem_alerts.json": len(siem)}
pd.DataFrame({"records_read": loaded, "documented_approx": documented})

,records_read,documented_approx
tenants.csv,6,6
agents.csv,22,22
tools.csv,18,18
model_pricing.csv,5,5
sandbox_policies.json,10,10
agent_activity.parquet,937752,940000
network_flows.csv,42392,42000
sandbox_audit.log,184005,184000
siem_alerts.json,3754,3800


In [5]:
# CSV line numbers are row index + 2 only if no field holds an embedded newline: verify
for name, df in [("tenants.csv", tenants), ("agents.csv", agents), ("tools.csv", tools),
                 ("model_pricing.csv", pricing), ("network_flows.csv", flows)]:
    lines = int(inv.loc[inv.file == name, "lines"].iloc[0])
    print(f"{name:20s} lines={lines:6d} rows+header={len(df) + 1:6d} ok={lines == len(df) + 1}")

tenants.csv          lines=     7 rows+header=     7 ok=True
agents.csv           lines=    23 rows+header=    23 ok=True
tools.csv            lines=    19 rows+header=    19 ok=True
model_pricing.csv    lines=     6 rows+header=     6 ok=True
network_flows.csv    lines= 42393 rows+header= 42393 ok=True


## 2. `tenants.csv`

In [6]:
profile_frame(tenants)

,column,dtype,nulls,null_rate,empty_str,distinct,top_values
0,tenant_id,str,0,0.0000,0,6,"tn_001 (1), tn_002 (1), tn_003 (1)"
1,tenant_name,str,0,0.0000,0,6,"Zephyr Logistics (1), Helios Care Group (1), NILEBANK Ltd. (1)"
2,industry,str,0,0.0000,0,5,"finance (2), logistics (1), healthcare (1)"
3,plan_tier,str,0,0.0000,0,3,"enterprise (3), growth (2), starter (1)"
4,region,str,0,0.0000,0,4,"eu-west-1 (2), me-south-1 (2), us-east-1 (1)"
5,onboarded_at,str,0,0.0000,0,5,"2024-11-11T08:15:00Z (2), 2025-06-02T09:00:00Z (1), 2025-03-18T14:30:00Z (1)"
6,is_active,str,0,0.0000,0,2,"true (5), false (1)"
7,security_contact,str,1,0.1667,0,4,"ciso@nilebank.example (2), soc@zephyr-logistics.example (1), security@helios-care.exam..."


In [7]:
TENANT_ENUMS = {"industry": ["finance", "healthcare", "logistics", "retail", "public_sector"],
                "plan_tier": ["starter", "growth", "enterprise"],
                "region": ["eu-west-1", "us-east-1", "me-south-1"],
                "is_active": ["true", "false"]}
for col, allowed in TENANT_ENUMS.items():
    bad = enum_violations(tenants[col], allowed)
    print(f"{col:10s} outside enum: {dict(bad)}")
print("null security_contact:", tenants.loc[tenants.security_contact.isna(), "tenant_id"].tolist())

industry   outside enum: {}
plan_tier  outside enum: {}
region     outside enum: {'eu-central-9': np.int64(1)}
is_active  outside enum: {}
null security_contact: ['tn_005']


In [8]:
# entity resolution: one company under two tenant ids?
norm = (tenants.tenant_name.str.lower()
        .str.replace(r"\b(ltd|inc|llc|plc)\b\.?", "", regex=True)
        .str.replace(r"[^a-z0-9]", "", regex=True))
view = tenants.assign(name_norm=norm)[["tenant_id", "tenant_name", "name_norm", "industry", "region",
                                        "onboarded_at", "security_contact"]]
same_company = view[view.duplicated(["name_norm"], keep=False)
                    | view.duplicated(["security_contact", "onboarded_at"], keep=False) & view.security_contact.notna()]
same_company

,tenant_id,tenant_name,name_norm,industry,region,onboarded_at,security_contact
2,tn_003,NILEBANK Ltd.,nilebank,finance,me-south-1,2024-11-11T08:15:00Z,ciso@nilebank.example
3,tn_004,nilebank,nilebank,finance,me-south-1,2024-11-11T08:15:00Z,ciso@nilebank.example


In [9]:
seen = act.groupby("tenant_id").agg(rows=("event_id", "size"), first_ingest=("ingest_time", "min"),
                                     last_ingest=("ingest_time", "max"))
tenants[["tenant_id", "is_active"]].merge(seen, left_on="tenant_id", right_index=True, how="left")

,tenant_id,is_active,rows,first_ingest,last_ingest
0,tn_001,true,578386.0,2026-01-31T00:00:35.885Z,2026-04-01T00:16:52.265Z
1,tn_002,true,122522.0,2026-01-31T00:10:19.610Z,2026-03-31T23:41:09.481Z
2,tn_003,true,124284.0,2026-01-31T00:05:55.905Z,2026-03-31T23:47:58.487Z
3,tn_004,true,NaN,NaN,NaN
4,tn_005,true,96781.0,2026-01-31T00:21:21.078Z,2026-04-01T00:10:03.207Z
5,tn_006,false,15043.0,2026-01-31T00:24:51.130Z,2026-02-24T23:45:03.016Z


In [10]:
F.add("T1", "tenants.csv", "tn_003 'NILEBANK Ltd.' and tn_004 'nilebank' are the same company",
      "normalised name + identical security_contact/onboarded_at", len(same_company),
      "entity-resolve tn_004 -> tn_003 (tn_004 has no activity)")
F.add("T2", "tenants.csv", "region 'eu-central-9' is not in the region enum",
      "enum check", int(enum_violations(tenants.region, TENANT_ENUMS["region"]).sum()),
      "set region null, quality flag INVALID_ENUM_REGION")
F.add("T3", "tenants.csv", "security_contact missing", "null check", int(tenants.security_contact.isna().sum()),
      "keep null (nullable attribute)")
F.add("T4", "tenants.csv", "inactive tenant tn_006 still has historical traffic", "join to activity",
      int(seen.loc["tn_006", "rows"]) if "tn_006" in seen.index else 0, "keep: documented as legitimate history")

## 3. `agents.csv` (natural key `(agent_id, release_version)`, Type-2 dimension)

In [11]:
profile_frame(agents)

,column,dtype,nulls,null_rate,empty_str,distinct,top_values
0,agent_id,str,0,0.0,0,14,"ag_007 (3), ag_004 (2), ag_002 (2)"
1,release_version,str,0,0.0,0,16,"1.0.0 (5), 2.2.0 (2), 1.1.0 (2)"
2,agent_name,str,0,0.0,0,15,"ops-runbook-agent (2), support-desk-agent (2), code-review-agent (2)"
3,agent_type,str,0,0.0,0,5,"support (6), research (6), analytics (5)"
4,model_id,str,0,0.0,0,4,"mdl_kst_lite_1 (10), mdl_kst_pro_1 (8), mdl_kst_pro_2 (2)"
5,tenant_id,str,0,0.0,0,5,"tn_001 (7), tn_003 (7), tn_002 (4)"
6,sandbox_profile_id,str,0,0.0,0,7,"sp_support_std (6), sp_research_std (6), sp_analytics_std (4)"
7,owner_team,str,0,0.0,0,13,"cx-platform (3), growth-intel (3), sre (2)"
8,autonomy_level,str,0,0.0,0,3,"semi_autonomous (9), supervised (9), autonomous (4)"
9,max_tool_calls_per_session,str,0,0.0,0,7,"10 (6), 25 (4), 20 (4)"


In [12]:
AGENT_ENUMS = {"agent_type": ["research", "support", "ops", "analytics", "coding"],
               "autonomy_level": ["supervised", "semi_autonomous", "autonomous"],
               "is_deprecated": ["true", "false"]}
for col, allowed in AGENT_ENUMS.items():
    print(f"{col:15s} outside enum: {dict(enum_violations(agents[col], allowed))}")
SEMVER = r"\d+\.\d+\.\d+"
print("non-semver release_version:", agents.loc[~agents.release_version.str.fullmatch(SEMVER), "release_version"].tolist())
print("raw natural-key duplicates:", int(agents.duplicated(["agent_id", "release_version"]).sum()))
agents_norm_ver = agents.release_version.str.lstrip("vV")
dup_mask = agents.assign(v=agents_norm_ver).duplicated(["agent_id", "v"], keep=False)
agents[dup_mask]

agent_type      outside enum: {}
autonomy_level  outside enum: {}
is_deprecated   outside enum: {}
non-semver release_version: ['v2.3.1']
raw natural-key duplicates: 0


,source_file,source_line,agent_id,release_version,agent_name,agent_type,model_id,tenant_id,sandbox_profile_id,owner_team,autonomy_level,max_tool_calls_per_session,launch_date,is_deprecated
12,agents.csv,14,ag_007,2.3.1,market-research-agent,research,mdl_kst_pro_1,tn_003,sp_research_std,growth-intel,semi_autonomous,25,2026-02-10,false
21,agents.csv,23,ag_007,v2.3.1,Market Research Agent,research,mdl_kst_pro_1,tn_003,sp_research_std,growth-intel,autonomous,25,2026-02-10,false


In [13]:
pair = agents[dup_mask].drop(columns=["source_file", "source_line"]).T
pair[pair.nunique(axis=1) > 1]          # the attributes on which the two 'versions' disagree

,12,21
release_version,2.3.1,v2.3.1
agent_name,market-research-agent,Market Research Agent
autonomy_level,semi_autonomous,autonomous


In [14]:
print("tenant_id not in tenants:", dict(fk_orphans(agents.tenant_id, tenants.tenant_id)))
print("sandbox_profile_id not in policies:", dict(fk_orphans(agents.sandbox_profile_id, policies.sandbox_profile_id)))
print("model_id not in pricing:", dict(fk_orphans(agents.model_id, pricing.model_id)))
print("distinct agents:", agents.agent_id.nunique(), "| rows:", len(agents),
      "| distinct (agent, normalised version):", agents.assign(v=agents_norm_ver)[["agent_id", "v"]].drop_duplicates().shape[0])

tenant_id not in tenants: {}
sandbox_profile_id not in policies: {'sp_missing_profile': np.int64(1)}
model_id not in pricing: {}
distinct agents: 14 | rows: 22 | distinct (agent, normalised version): 21


In [15]:
F.add("AG1", "agents.csv", "ag_007 'v2.3.1' duplicates 2.3.1 with a different agent_name and autonomy_level",
      "strip 'v' prefix, then duplicate natural key", int(dup_mask.sum()) - 1,
      "drop the v-prefixed row, keep canonical 2.3.1 attributes; record in entity_resolution")
F.add("AG2", "agents.csv", "ag_019 points to sandbox profile 'sp_missing_profile' that does not exist",
      "FK check against sandbox_policies", int(fk_orphans(agents.sandbox_profile_id, policies.sandbox_profile_id).sum()),
      "keep agent; policy_version_applied null and quality flag NO_POLICY_PROFILE for its records")

## 4. `tools.csv` (natural key `(tool_id, tool_version)`)

In [16]:
profile_frame(tools)

,column,dtype,nulls,null_rate,empty_str,distinct,top_values
0,tool_id,str,0,0.0,0,12,"tl_web_fetch (3), tl_vector_search (3), tl_sql_query (2)"
1,tool_version,str,0,0.0,0,14,"1.0.0 (4), 2.0.1 (2), 1.2.0 (1)"
2,tool_name,str,0,0.0,0,14,"vector_search (3), sql_query (2), code_exec (2)"
3,tool_category,str,0,0.0,0,7,"retrieval (7), database (3), shell (2)"
4,risk_tier,str,0,0.0,0,4,"low (9), high (4), medium (3)"
5,requires_approval,str,0,0.0,0,2,"false (13), true (5)"
6,owner_team,str,0,0.0,0,4,"platform-tools (12), data-platform (2), cx-platform (2)"


In [17]:
TOOL_ENUMS = {"tool_category": ["retrieval", "database", "shell", "network", "filesystem", "messaging", "code_exec"],
              "risk_tier": ["low", "medium", "high", "critical"], "requires_approval": ["true", "false"]}
for col, allowed in TOOL_ENUMS.items():
    print(f"{col:18s} outside enum: {dict(enum_violations(tools[col], allowed))}")
body = tools.drop(columns=["source_file", "source_line"])
print("exact duplicate rows:", int(body.duplicated().sum()))
print("natural-key duplicates:", int(body.duplicated(["tool_id", "tool_version"]).sum()))
tools[body.duplicated(keep=False)]

tool_category      outside enum: {}
risk_tier          outside enum: {}
requires_approval  outside enum: {}
exact duplicate rows: 1
natural-key duplicates: 1


,source_file,source_line,tool_id,tool_version,tool_name,tool_category,risk_tier,requires_approval,owner_team
3,tools.csv,5,tl_vector_search,2.0.1,vector_search,retrieval,low,false,platform-tools
4,tools.csv,6,tl_vector_search,2.0.1,vector_search,retrieval,low,false,platform-tools


In [18]:
per_tool = tools.groupby("tool_id").agg(versions=("tool_version", "nunique"), names=("tool_name", "unique"),
                                         categories=("tool_category", "unique"))
per_tool[(per_tool.names.str.len() > 1) | (per_tool.categories.str.len() > 1)]

,versions,names,categories
tool_id,,,
tl_code_exec,2,[code_exec],"[code_exec, shell]"
tl_web_fetch,3,"[web_fetch, http_get, webfetch]",[retrieval]


In [19]:
F.add("TL1", "tools.csv", "tl_vector_search 2.0.1 appears twice (identical rows)", "duplicated() on all columns",
      int(body.duplicated().sum()), "deduplicate on (tool_id, tool_version)")
F.add("TL2", "tools.csv", "tl_web_fetch is named web_fetch / http_get / webfetch across versions",
      "nunique(tool_name) per tool_id", 3, "build alias map tool_name -> tool_id for entity resolution")
F.add("TL3", "tools.csv", "tl_code_exec category is code_exec in 0.9.1 but shell in 0.9.0",
      "nunique(tool_category) per tool_id", 1, "keep per-version category, quality flag; document")

## 5. `model_pricing.csv` (natural key `(model_id, effective_from)`)

In [20]:
pricing.drop(columns=['source_file'])

,source_line,model_id,model_family,input_price_per_1k,output_price_per_1k,currency,effective_from,effective_to
0,2,mdl_kst_pro_1,kestrel-pro,0.003000,0.015000,USD,2026-01-01T00:00:00Z,NaN
1,3,mdl_kst_pro_1,kestrel-pro,0.002500,0.012000,USD,2026-03-01T00:00:00Z,NaN
2,4,mdl_kst_pro_2,kestrel-pro,0.002000,0.010000,USD,2026-03-01T00:00:00Z,NaN
3,5,mdl_kst_lite_1,kestrel-lite,0.000500,0.001500,USD,2026-01-01T00:00:00Z,NaN
4,6,mdl_kst_nano_1,kestrel-nano,NaN,NaN,USD,2026-02-01T00:00:00Z,NaN


In [21]:
p = pricing.copy()
p["effective_from_ts"] = pd.to_datetime(p.effective_from, utc=True, format="ISO8601")
print("null prices:", p.loc[p.input_price_per_1k.isna() | p.output_price_per_1k.isna(), "model_id"].tolist())
print("effective_to filled:", int(p.effective_to.notna().sum()), "of", len(p))
print("currencies:", p.currency.unique().tolist())
print("6-decimal money format:", bool(p.input_price_per_1k.dropna().str.fullmatch(r"\d+\.\d{6}").all()))
print("model_ids used by agents but not priced:", dict(fk_orphans(agents.model_id, p.model_id)))

null prices: ['mdl_kst_nano_1']
effective_to filled: 0 of 5
currencies: ['USD']
6-decimal money format: True
model_ids used by agents but not priced: {}


In [22]:
# does the price table cover every turn? compare first turn per model with its first effective_from
turns = act[act.event_type == "llm.turn"]
turn_t = pd.to_datetime(turns.event_time, utc=True, format="ISO8601", errors="coerce")   # diagnostic parse, ISO rows only
first_price = p.groupby("model_id").effective_from_ts.min()
cov = (turns.assign(t=turn_t).groupby("model_id").t.agg(first_turn="min", turns="size")
       .join(first_price.rename("first_effective_from")))
cov["turns_before_first_price"] = [int(((turns.model_id == m) & (turn_t < cov.loc[m, "first_effective_from"])).sum())
                                   for m in cov.index]
cov

,first_turn,turns,first_effective_from,turns_before_first_price
model_id,,,,
mdl_kst_lite_1,2026-01-31 00:00:34.757000+00:00,104209,2026-01-01 00:00:00+00:00,0
mdl_kst_nano_1,2026-01-31 00:05:57.092000+00:00,29084,2026-02-01 00:00:00+00:00,451
mdl_kst_pro_1,2026-01-31 00:00:50.463000+00:00,268027,2026-01-01 00:00:00+00:00,0
mdl_kst_pro_2,2026-03-05 00:11:49.575000+00:00,11873,2026-03-01 00:00:00+00:00,0


In [23]:
F.add("P1", "model_pricing.csv", "mdl_kst_nano_1 has empty input/output prices", "null check", 1,
      "cost_usd null for nano turns, quality flag PRICE_MISSING; declared in the cost policy")
F.add("P2", "model_pricing.csv", "effective_to is never filled", "null check", int(p.effective_to.isna().sum()),
      "derive effective_to = next effective_from of the same model (open-ended for the last)")
F.add("P3", "model_pricing.csv", "nano turns occur before nano's first effective_from (2026-02-01)",
      "first turn per model vs first effective_from", int(cov.turns_before_first_price.sum()),
      "no price effective at turn time -> cost null + flag")

## 6. `sandbox_policies.json` (nested; flattened to one row per policy version)

`has_network_block = False` is kept on purpose: per the spec a version **without** a network block allows no egress
at all, which flattening to null columns would otherwise make indistinguishable from "no restriction".

In [24]:
policies.drop(columns=['source_file'])

,source_line,sandbox_profile_id,policy_version,valid_from,valid_to,allowed_tools,escalation_allowed,network_egress_allowlist,network_denied_ports,network_max_egress_mb,filesystem_read_scope,filesystem_write_scope,resources_max_runtime_s,resources_max_memory_mb,has_network_block
0,3,sp_research_std,2,2026-01-01T00:00:00Z,2026-02-01T00:00:00Z,"[tl_web_fetch, tl_vector_search]",False,"[api.kestrel.ai, *.public-data.example]","[22, 23, 445, 3389]",25.0,[/workspace],[/workspace/out],600,1024,True
1,39,sp_research_std,3,2026-01-25T00:00:00Z,NaN,"[tl_web_fetch, tl_vector_search, tl_summarize]",False,"[api.kestrel.ai, *.public-data.example, research.marketwire.example]","[22, 23, 445, 3389]",50.0,[/workspace],[/workspace/out],900,2048,True
2,77,sp_support_std,1,2026-01-01T00:00:00Z,2026-03-01T00:00:00Z,"[tl_vector_search, tl_send_email, tl_summarize]",False,"[api.kestrel.ai, mail.kestrel.ai]","[22, 445]",10.0,[/workspace],[],300,512,True
3,110,sp_support_std,2,2026-03-01T00:00:00Z,NaN,"[tl_vector_search, tl_send_email, tl_summarize, tl_sql_query]",False,"[api.kestrel.ai, mail.kestrel.ai]","[22, 445]",10.0,[/workspace],[],300,512,True
4,144,sp_ops_elevated,3,2025-12-01T00:00:00Z,2026-02-15T00:00:00Z,"[tl_shell_exec, tl_sql_query, tl_file_write]",True,"[api.kestrel.ai, *.status.kestrel.ai]",[23],100.0,"[/workspace, /etc/runbooks]",[/workspace/out],1800,4096,True
5,179,sp_ops_elevated,4,2026-02-15T00:00:00Z,NaN,"[tl_shell_exec, tl_sql_query, tl_file_write, tl_web_fetch]",True,NaN,NaN,NaN,"[/workspace, /etc/runbooks]",[/workspace/out],1800,4096,False
6,205,sp_ops_std,1,2026-01-01T00:00:00Z,NaN,"[tl_shell_exec, tl_sql_query, tl_file_write]",False,"[api.kestrel.ai, *.status.kestrel.ai]","[22, 23, 445, 3389]",20.0,[/workspace],[/workspace/out],900,1024,True
7,242,sp_analytics_std,1,2026-01-01T00:00:00Z,2026-03-15T00:00:00Z,"[tl_sql_query, tl_summarize, tl_file_write]",False,"[api.kestrel.ai, *.public-data.example]","[22, 23, 445, 3389]",50.0,[/workspace],[/workspace/out],1200,4096,True
8,279,sp_analytics_std,2,2026-03-15T00:00:00Z,NaN,"[tl_sql_query, tl_summarize, tl_file_write, tl_code_exec]",False,"[api.kestrel.ai, *.public-data.example]","[22, 23, 445, 3389]",50.0,[/workspace],[/workspace/out],1200,4096,True
9,317,sp_coding_std,1,2026-01-01T00:00:00Z,NaN,"[tl_code_exec, tl_file_write, tl_web_fetch, tl_vector_search]",False,"[api.kestrel.ai, *.pkg-mirror.example]","[22, 23, 445, 3389]",200.0,[/workspace],[/workspace],1800,4096,True


In [25]:
raw_keys = pd.DataFrame([{k: (k in r) for k in ["allowed_tools", "network", "filesystem", "resources",
                                                "escalation_allowed", "valid_to"]} | {"profile": r["sandbox_profile_id"],
                         "version": r["policy_version"]} for r in rd.read_sandbox_policies_raw()])
raw_keys[~raw_keys.drop(columns=["profile", "version"]).all(axis=1)]

,allowed_tools,network,filesystem,resources,escalation_allowed,valid_to,profile,version
5,True,False,True,True,True,True,sp_ops_elevated,4


In [26]:
pol = policies.assign(valid_from_ts=pd.to_datetime(policies.valid_from, utc=True),
                      valid_to_ts=pd.to_datetime(policies.valid_to, utc=True))
ov = interval_overlaps(pol, "sandbox_profile_id", "valid_from_ts", "valid_to_ts")
pd.DataFrame([{"profile": r.sandbox_profile_id, "version_a": r.a["policy_version"], "version_b": r.b["policy_version"],
               "overlap_from": r.overlap_from, "overlap_to": r.overlap_to} for r in ov.itertuples()])

,profile,version_a,version_b,overlap_from,overlap_to
0,sp_research_std,2,3,2026-01-25 00:00:00+00:00,2026-02-01 00:00:00+00:00


In [27]:
wild = pol.explode("network_egress_allowlist").dropna(subset=["network_egress_allowlist"])
print("wildcard allowlist entries:", sorted(wild.loc[wild.network_egress_allowlist.str.contains(r"\*"),
                                                      "network_egress_allowlist"].unique()))
print("example: host_matches('census.public-data.example', ['*.public-data.example']) =",
      host_matches("census.public-data.example", ["*.public-data.example"]))
print("tools in allowlists not in tools.csv:",
      sorted(set(pol.allowed_tools.explode()) - set(tools.tool_id)))

wildcard allowlist entries: ['*.pkg-mirror.example', '*.public-data.example', '*.status.kestrel.ai']
example: host_matches('census.public-data.example', ['*.public-data.example']) = True
tools in allowlists not in tools.csv: []


In [28]:
F.add("PO1", "sandbox_policies.json", "sp_research_std v2 (to 2026-02-01) and v3 (from 2026-01-25) overlap",
      "pairwise interval-overlap check per profile", len(ov),
      "resolve with evidence from the audit log (finding PO1b, section 9.1)")
F.add("PO2", "sandbox_policies.json", "sp_ops_elevated v4 has no network block (= no egress allowed)",
      "key-presence check on raw JSON", int((~pol.has_network_block).sum()),
      "keep has_network_block=False; every egress under v4 is a violation")
F.add("PO3", "sandbox_policies.json", "egress allowlists contain wildcards", "regex on exploded allowlist",
      int(wild.network_egress_allowlist.str.contains(r"\*").sum()), "fnmatch-style host matching, never ==")

## 7. `agent_activity.parquet`

### 7.1 Schema check against the Dataset Description contract

In [29]:
sc = rd.activity_schema_check()
print("columns:", len(sc), "| mismatches:", int((~sc.ok).sum()))
sc[~sc.ok] if (~sc.ok).any() else sc.head(3)

columns: 30 | mismatches: 0


,column,expected,actual,ok
0,agent_id,string,string,True
1,bytes_out,int64,int64,True
2,content_source,string,string,True


### 7.2 Column profile

In [30]:
profile_frame(act)

,column,dtype,nulls,null_rate,empty_str,distinct,top_values
0,event_id,str,0,0.0000,0,930325,"ev_0017604 (2), ev_0666844 (2), ev_0449411 (2)"
1,event_time,str,0,0.0000,0,929319,"2026-02-30T10:00:00Z (366), not-a-time (347), 2026-03-14T11:07:26.504Z (3)"
2,ingest_time,str,0,0.0000,0,937393,"2026-02-13T15:26:07.025Z (2), 2026-03-05T07:19:19.976Z (2), 2026-03-29T22:34:46.601Z (2)"
3,sequence_number,Int64,302,0.0003,0,18,"2 (101197), 4 (101179), 1 (101141)"
4,event_type,str,0,0.0000,0,7,"llm.turn (413526), tool.call (323750), session.start (101141)"
5,session_id,str,0,0.0000,0,100533,"ss_021442 (19), ss_092744 (19), ss_093940 (18)"
6,tenant_id,str,0,0.0000,0,6,"tn_001 (578386), tn_003 (124284), tn_002 (122522)"
7,agent_id,str,0,0.0000,0,16,"ag_004 (443889), ag_001 (55966), ag_019 (50419)"
8,release_version,str,0,0.0000,0,18,"3.0.2 (299235), 3.0.1 (144654), 1.0.0 (138339)"
9,sandbox_instance_id,str,0,0.0000,0,43,"sbx_01a0 (149405), sbx_01ad (147518), sbx_01ba (146966)"


### 7.3 Fill pattern by `event_type`

The row's `event_type` decides which columns are filled (spec 3.5). Share of non-null values per type:

In [31]:
fill = act.drop(columns=["source_file", "source_line"]).notna().groupby(act.event_type).mean().T.round(4)
fill

event_type,agent.telepathy,llm.thought,llm.turn,session.end,session.error,session.start,tool.call
event_id,1.0,1.0,1.0000,1.0000,1.0,1.0000,1.0000
event_time,1.0,1.0,1.0000,1.0000,1.0,1.0000,1.0000
ingest_time,1.0,1.0,1.0000,1.0000,1.0,1.0000,1.0000
sequence_number,0.0,0.0,1.0000,1.0000,1.0,1.0000,1.0000
event_type,1.0,1.0,1.0000,1.0000,1.0,1.0000,1.0000
session_id,1.0,1.0,1.0000,1.0000,1.0,1.0000,1.0000
tenant_id,1.0,1.0,1.0000,1.0000,1.0,1.0000,1.0000
agent_id,1.0,1.0,1.0000,1.0000,1.0,1.0000,1.0000
release_version,1.0,1.0,1.0000,1.0000,1.0,1.0000,1.0000
sandbox_instance_id,1.0,1.0,1.0000,1.0000,1.0,1.0000,1.0000


In [32]:
EVENT_TYPES = ["session.start", "llm.turn", "tool.call", "session.end", "session.error"]
REQUIRED = {"*": ["event_id", "event_time", "ingest_time", "sequence_number", "event_type", "session_id",
                  "tenant_id", "agent_id", "release_version", "sandbox_instance_id", "status"],
            "session.start": ["task_id", "task_category", "expected_tool_categories"],
            "llm.turn": ["turn_index", "model_id", "content_source", "input_tokens", "output_tokens",
                         "finish_reason", "latency_ms"],
            "tool.call": ["tool_id", "tool_version", "finished_at", "retry_number", "target_resource", "latency_ms"],
            "session.end": ["latency_ms"], "session.error": ["latency_ms"]}
rows = []
for et in EVENT_TYPES:
    sub = act[act.event_type == et]
    for col in REQUIRED["*"] + REQUIRED[et]:
        n = int(sub[col].isna().sum())
        if n:
            rows.append({"event_type": et, "column": col, "missing": n})
missing_required = pd.DataFrame(rows)
missing_required

,event_type,column,missing
0,session.start,task_id,333
1,session.start,task_category,333
2,session.start,expected_tool_categories,333
3,llm.turn,status,400
4,llm.turn,turn_index,333
5,llm.turn,model_id,333
6,llm.turn,content_source,333
7,llm.turn,input_tokens,333
8,llm.turn,output_tokens,4449
9,llm.turn,finish_reason,333


### 7.4 Enumerations

In [33]:
print("event_type outside enum:", dict(enum_violations(act.event_type, EVENT_TYPES)))
pd.crosstab(act.event_type, act.status.fillna("<null>"))

event_type outside enum: {'agent.telepathy': np.int64(152), 'llm.thought': np.int64(150)}


status,<null>,denied,error,failure,ok,succeeded,success,timeout
event_type,,,,,,,,
agent.telepathy,0,0,0,0,152,0,0,0
llm.thought,0,0,0,0,150,0,0,0
llm.turn,400,0,0,0,413126,0,0,0
session.end,0,0,0,0,91636,0,0,0
session.error,0,0,7397,0,0,0,0,0
session.start,0,0,0,0,101141,0,0,0
tool.call,338,4551,0,14762,0,160,293218,10721


In [34]:
ACT_ENUMS = {"content_source": ["user", "system", "retrieved_web", "tool_output"],
             "finish_reason": ["stop", "length", "tool_call", "content_filter"],
             "error_class": ["TIMEOUT", "AUTH_DENIED", "RATE_LIMIT", "POLICY_DENIED", "TARGET_UNREACHABLE", "INVALID_ARGS"],
             "task_category": ["market_research", "ticket_triage", "report_generation", "data_query", "code_review",
                               "incident_response"]}
for col, allowed in ACT_ENUMS.items():
    print(f"{col:15s} outside enum: {dict(enum_violations(act[col], allowed))} | values: {sorted(act[col].dropna().unique())}")
INDICATORS = ["instruction_override", "role_confusion", "exfil_request", "encoded_payload", "tool_coercion", "policy_probe"]
tok = act.loc[act.event_type == "llm.turn", "injection_indicators"].dropna()
print("injection_indicators: empty string =", int((tok == "").sum()), "| null =",
      int(act.loc[act.event_type == "llm.turn", "injection_indicators"].isna().sum()))
print("indicator tokens outside enum:", dict(enum_violations(tok[tok != ""].str.split("|").explode(), INDICATORS)))
cats = act.expected_tool_categories.dropna().str.split("|").explode()
print("expected_tool_categories tokens outside enum:", dict(enum_violations(cats, TOOL_ENUMS["tool_category"])))

content_source  outside enum: {} | values: ['retrieved_web', 'system', 'tool_output', 'user']
finish_reason   outside enum: {} | values: ['tool_call']
error_class     outside enum: {} | values: ['AUTH_DENIED', 'INVALID_ARGS', 'POLICY_DENIED', 'RATE_LIMIT', 'TARGET_UNREACHABLE', 'TIMEOUT']
task_category   outside enum: {} | values: ['code_review', 'data_query', 'incident_response', 'market_research', 'report_generation', 'ticket_triage']
injection_indicators: empty string = 402841 | null = 333
indicator tokens outside enum: {}
expected_tool_categories tokens outside enum: {}


In [35]:
tc = act[act.event_type == "tool.call"]
pd.crosstab(tc.error_class.fillna("<null>"), tc.status.fillna("<null>"))

status,<null>,denied,failure,succeeded,success,timeout
error_class,,,,,,
<null>,306,0,0,143,293218,0
AUTH_DENIED,3,0,4172,2,0,0
INVALID_ARGS,4,0,4306,2,0,0
POLICY_DENIED,7,4551,0,4,0,0
RATE_LIMIT,5,0,4232,3,0,0
TARGET_UNREACHABLE,1,0,2052,1,0,0
TIMEOUT,12,0,0,5,0,10721


### 7.5 Timestamps

`event_time`, `ingest_time` and `finished_at` are raw strings. First the *shapes* (digits → 9) show how many
formats coexist; then each event_time format is classified and its convention established against `ingest_time`
(which has a single clean ISO-UTC format): if a format is UTC, its lateness `ingest − event` must look like the
ISO rows' lateness (≈1 s), not like a whole-hour offset.

In [36]:
for col in ["event_time", "ingest_time", "finished_at"]:
    print(f"--- {col}"); print(value_shapes(act[col]).to_string())

--- event_time
event_time
9999-99-99A99:99:99.999A    922618
9999999999999                14029
99/99/9999 99:99:99            392
9999-99-99A99:99:99A           366
aaa-a-aaaa                     347
--- ingest_time
ingest_time
9999-99-99A99:99:99.999A    937752
--- finished_at
finished_at
9999-99-99A99:99:99.999A    323417


In [37]:
et = act.event_time
FORMATS = {"iso_ms": r"\d{4}-\d\d-\d\dT\d\d:\d\d:\d\d\.\d{3}Z", "iso_s": r"\d{4}-\d\d-\d\dT\d\d:\d\d:\d\dZ",
           "epoch_ms": r"\d{13}", "dmy_hms": r"\d\d/\d\d/\d{4} \d\d:\d\d:\d\d"}
fmt = pd.Series("other", index=act.index)
for name, rx in FORMATS.items():
    fmt[et.str.fullmatch(rx)] = name
ingest = pd.to_datetime(act.ingest_time, utc=True, format="ISO8601")

def diag_parse(fmt_name):
    s = et[fmt == fmt_name]
    if fmt_name in ("iso_ms", "iso_s"):
        out = pd.to_datetime(s, utc=True, format="ISO8601", errors="coerce")
    elif fmt_name == "epoch_ms":
        out = pd.to_datetime(s.astype("int64"), unit="ms", utc=True)
    elif fmt_name == "dmy_hms":
        out = pd.to_datetime(s, format="%d/%m/%Y %H:%M:%S", utc=True, errors="coerce")
    else:
        out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns, UTC]")
    return out.astype("datetime64[ms, UTC]")

event_ts = pd.concat([diag_parse(f) for f in ["iso_ms", "iso_s", "epoch_ms", "dmy_hms", "other"]]).sort_index()
lateness = (ingest - event_ts).dt.total_seconds()
summary = pd.DataFrame({"rows": fmt.value_counts(),
                        "unparseable": event_ts.isna().groupby(fmt).sum(),
                        "lateness_p50_s": lateness.groupby(fmt).median().round(3),
                        "lateness_p99_s": lateness.groupby(fmt).quantile(0.99).round(1)})
summary

,rows,unparseable,lateness_p50_s,lateness_p99_s
dmy_hms,392,0,1.410,1078.6
epoch_ms,14029,0,0.916,840.8
iso_ms,922618,0,0.915,982.3
iso_s,366,366,NaN,NaN
other,347,347,NaN,NaN


In [38]:
for f in ["iso_s", "dmy_hms", "other"]:
    print(f, "samples:", et[fmt == f].value_counts().head(4).to_dict())
d = et[fmt == "dmy_hms"]
first, second = d.str.slice(0, 2).astype(int), d.str.slice(3, 5).astype(int)
print(f"dmy_hms: first field > 12 in {int((first > 12).sum())} rows, second field > 12 in {int((second > 12).sum())} rows "
      "-> day-first is the only consistent reading")
mdy = pd.to_datetime(d, format="%m/%d/%Y %H:%M:%S", errors="coerce")
print("reading it month-first instead would fail on", int(mdy.isna().sum()), "rows")

iso_s samples: {'2026-02-30T10:00:00Z': 366}
dmy_hms samples: {'18/03/2026 18:07:33': 2, '13/03/2026 08:20:55': 2, '06/03/2026 14:20:19': 2, '02/02/2026 07:47:44': 2}
other samples: {'not-a-time': 347}
dmy_hms: first field > 12 in 250 rows, second field > 12 in 0 rows -> day-first is the only consistent reading
reading it month-first instead would fail on 250 rows


In [39]:
lat = lateness.dropna()
print(f"lateness over {len(lat):,} parsed rows: negative={int((lat < 0).sum())}, >10 s={int((lat > 10).sum())} "
      f"({(lat > 10).mean():.2%}), >60 s={int((lat > 60).sum())}, max={lat.max():.0f} s")
print("event_time range:", event_ts.min(), "->", event_ts.max())
print("ingest_time range:", ingest.min(), "->", ingest.max())
fin = pd.to_datetime(act.finished_at, utc=True, format="ISO8601", errors="coerce")
dur_call = (fin - event_ts).dt.total_seconds()[act.event_type == "tool.call"].dropna()
print("tool.call finished_at before its start:", int((dur_call < 0).sum()))

lateness over 937,039 parsed rows: negative=0, >10 s=21256 (2.27%), >60 s=20613, max=3280 s
event_time range: 2026-01-31 00:00:33.753000+00:00 -> 2026-03-31 23:54:31.573000+00:00
ingest_time range: 2026-01-31 00:00:35.885000+00:00 -> 2026-04-01 00:16:52.265000+00:00
tool.call finished_at before its start: 1618


In [40]:
F.add("AC1", "agent_activity.parquet", "event_time stored in 4+ formats (ISO ms, epoch ms, dd/mm/yyyy, ISO without ms)",
      "value shapes (digits->9) of event_time", int((fmt != "iso_ms").sum()),
      "parse every format to UTC; non-ISO formats get quality flag NONSTANDARD_TIMESTAMP_FORMAT")
F.add("AC2", "agent_activity.parquet", "epoch-ms and dd/mm/yyyy event_times are UTC",
      "lateness vs ingest_time ~1 s for every format (no whole-hour offset); day field > 12 proves day-first",
      int(((fmt == "epoch_ms") | (fmt == "dmy_hms")).sum()), "interpret as UTC, epoch unit = ms, day-first")
F.add("AC3", "agent_activity.parquet", "unparseable event_time ('not-a-time', impossible date 2026-02-30)",
      "regex classification + strict parse", int(event_ts.isna().sum()), "reject MALFORMED_TIMESTAMP")
F.add("AC4", "agent_activity.parquet", "late events (ingest - event > 10 s), up to ~55 min",
      "lateness distribution", int((lat > 10).sum()),
      "accept + quality flag LATE (batch has no watermark; Kafka 60 s watermark is M2)")
F.add("AC4b", "agent_activity.parquet", "tool.call finished_at is earlier than its start (event_time)",
      "finished_at - event_time < 0 on parsed rows", int((dur_call < 0).sum()),
      "keep row; quality flag FINISHED_BEFORE_START (finished_at is not used by any M1 metric)")

### 7.6 Duplicates (delivery is at-least-once)

In [41]:
dup = act[act.event_id.duplicated(keep=False)]
print("rows sharing an event_id:", len(dup), "| distinct ids:", dup.event_id.nunique(),
      "| surplus copies:", int(act.event_id.duplicated().sum()))
cols = [c for c in act.columns if c not in ("ingest_time", "source_line")]
print("copies identical except ingest_time/source_line:", bool(dup.duplicated(cols, keep=False).all()))
gap = dup.assign(ing=ingest[dup.index]).groupby("event_id").ing.agg(lambda s: (s.max() - s.min()).total_seconds())
print("ingest gap between copies (s):"); print(gap.describe().round(2).to_string())

rows sharing an event_id: 14854 | distinct ids: 7427 | surplus copies: 7427
copies identical except ingest_time/source_line: True
ingest gap between copies (s):
count    7427.00
mean      300.76
std       174.09
min         1.08
25%       151.51
50%       300.76
75%       453.30
max       599.96


In [42]:
F.add("AC5", "agent_activity.parquet", "duplicate event_id (identical payload, different ingest_time)",
      "event_id.duplicated()", int(act.event_id.duplicated().sum()),
      "keep earliest ingest_time; other copies rejected DUPLICATE_EVENT_ID")

### 7.7 Referential integrity

In [43]:
print("tenant_id not in tenants:", dict(fk_orphans(act.tenant_id, tenants.tenant_id)))
print("agent_id not in agents:", dict(fk_orphans(act.agent_id, agents.agent_id)))
key = act.agent_id + "@" + act.release_version
key_norm = act.agent_id + "@" + act.release_version.str.lstrip("vV")
ref = agents.agent_id + "@" + agents.release_version
print("(agent, version) not in agents, raw:", dict(fk_orphans(key, ref)))
print("(agent, version) not in agents, after stripping 'v':", dict(fk_orphans(key_norm, ref)))

tenant_id not in tenants: {'tn_777': np.int64(736)}
agent_id not in agents: {'ag_999': np.int64(672), 'ag_321': np.int64(660)}
(agent, version) not in agents, raw: {'ag_999@9.9.9': np.int64(672), 'ag_321@9.9.9': np.int64(660), 'ag_007@v2.2.0': np.int64(359)}
(agent, version) not in agents, after stripping 'v': {'ag_999@9.9.9': np.int64(672), 'ag_321@9.9.9': np.int64(660)}


In [44]:
tc = act[act.event_type == "tool.call"]
print("tool_id not in tools.tool_id:", dict(fk_orphans(tc.tool_id, tools.tool_id)))
print("  ...of which are tool *names*:", sorted(set(fk_orphans(tc.tool_id, tools.tool_id).index) & set(tools.tool_name)))
print("(tool, version) not in tools:", dict(fk_orphans(tc.tool_id + "@" + tc.tool_version, tools.tool_id + "@" + tools.tool_version)))
print("model_id not in pricing:", dict(fk_orphans(act.model_id, pricing.model_id)))
own = agents.drop_duplicates("agent_id").set_index("agent_id").tenant_id
mismatch = act[act.agent_id.isin(own.index) & (act.tenant_id != act.agent_id.map(own))]
print("rows whose tenant differs from the agent's owning tenant:", dict(mismatch.tenant_id.value_counts()))

tool_id not in tools.tool_id: {'http_get': np.int64(490), 'webfetch': np.int64(490)}
  ...of which are tool *names*: ['http_get', 'webfetch']
(tool, version) not in tools: {'http_get@1.1.0': np.int64(490), 'webfetch@1.0.0': np.int64(490)}
model_id not in pricing: {}
rows whose tenant differs from the agent's owning tenant: {'tn_777': np.int64(736)}


In [45]:
F.add("AC6", "agent_activity.parquet", "unknown agents ag_999 / ag_321 (version 9.9.9)", "FK check vs agents.csv",
      int(act.agent_id.isin(["ag_999", "ag_321"]).sum()), "reject UNKNOWN_AGENT_ID")
F.add("AC7", "agent_activity.parquet", "unknown tenant tn_777 inside otherwise valid sessions",
      "FK check vs tenants.csv", int((act.tenant_id == "tn_777").sum()), "reject UNKNOWN_TENANT_ID")
F.add("AC8", "agent_activity.parquet", "release_version with 'v' prefix (v2.3.1, v2.2.0)",
      "FK check fails raw, passes after stripping 'v'", int(act.release_version.str.startswith("v").sum()),
      "normalise (strip v) + quality flag VERSION_PREFIX_STRIPPED")
F.add("AC9", "agent_activity.parquet", "tool.call tool_id holds a tool name (http_get, webfetch)",
      "FK check vs tool_id, match vs tool_name", int(tc.tool_id.isin(["http_get", "webfetch"]).sum()),
      "resolve via alias map to tl_web_fetch + quality flag TOOL_ALIAS_RESOLVED")

### 7.8 Invalid event types, missing fields, numeric ranges

In [46]:
print("invalid event types:", dict(enum_violations(act.event_type, EVENT_TYPES)))
numeric_ranges(act, ["sequence_number", "latency_ms", "turn_index", "input_tokens", "output_tokens",
                     "retry_number", "bytes_out"])

invalid event types: {'agent.telepathy': np.int64(152), 'llm.thought': np.int64(150)}


,column,count,min,p50,p99,max,negative,zero
0,sequence_number,937450,1,5.0,15.00,18,0,0
1,latency_ms,835310,-4000,2118.0,34185.00,109492,411,0
2,turn_index,413193,0,2.0,6.00,7,0,100827
3,input_tokens,413193,-12655,3286.0,10624.00,14576,830,0
4,output_tokens,409077,30,240.0,446.00,450,0,0
5,retry_number,323417,-1,0.0,1.00,2,321,314014
6,bytes_out,43300,128,1401.0,41673.71,65525,0,0


In [47]:
bad_type = ~act.event_type.isin(EVENT_TYPES)
status_null = act.status.isna() & ~bad_type
neg = (act.latency_ms < 0).fillna(False) | (act.input_tokens < 0).fillna(False) | (act.retry_number < 0).fillna(False)
F.add("AC10", "agent_activity.parquet", "event types outside the enum (agent.telepathy, llm.thought)",
      "enum check on event_type", int(bad_type.sum()), "reject INVALID_EVENT_TYPE")
F.add("AC11", "agent_activity.parquet", "status null on llm.turn / tool.call", "required-field null check",
      int(status_null.sum()), "reject MISSING_REQUIRED_FIELD")
F.add("AC12", "agent_activity.parquet", "status 'succeeded' is not in the tool-call status enum",
      "crosstab event_type x status", int((act.status == "succeeded").sum()),
      "map to 'success' + quality flag STATUS_NORMALISED")
F.add("AC13", "agent_activity.parquet", "negative latency_ms / input_tokens, retry_number = -1",
      "min/negative counts per numeric column", int(neg.sum()), "reject INVALID_NUMERIC_RANGE")
F.add("AC14", "agent_activity.parquet", "llm.turn with null output_tokens",
      "required-field null check", int((act.event_type.eq("llm.turn") & act.output_tokens.isna()).sum()),
      "keep row; declared missing-token policy = null-excluded; effect shown on one metric")

### 7.9 Session structure

A session runs `session.start` → alternating `llm.turn` / `tool.call` → one terminal row. Checked on a
*diagnostic* de-duplicated view (earliest ingest per event_id); the real rules are applied in the cleaning module.

In [48]:
d = act.assign(ingest_ts=ingest, event_ts=event_ts).sort_values("ingest_ts").drop_duplicates("event_id")
d = d[d.event_type.isin(EVENT_TYPES)]
per = d.groupby("session_id").agg(
    rows=("event_id", "size"),
    starts=("event_type", lambda s: int((s == "session.start").sum())),
    terminals=("event_type", lambda s: int(s.isin(["session.end", "session.error"]).sum())),
    max_seq=("sequence_number", "max"), distinct_seq=("sequence_number", "nunique"),
    tenants=("tenant_id", "nunique"), versions=("release_version", "nunique"))
print("sessions:", len(per))
print("starts per session:", per.starts.value_counts().to_dict())
print("terminal events per session:", per.terminals.value_counts().to_dict())
print("sessions without a start:", int((per.starts == 0).sum()), "| without a terminal:", int((per.terminals == 0).sum()))
print("sessions with a sequence gap (max_seq != rows):", int((per.max_seq != per.rows).sum()),
      "| with repeated sequence numbers:", int((per.distinct_seq != per.rows).sum()))
print("sessions mixing tenants:", int((per.tenants > 1).sum()), "| mixing release_version spellings:", int((per.versions > 1).sum()))

sessions: 100533
starts per session: {1: 100333, 0: 200}
terminal events per session: {1: 98285, 0: 2248}
sessions without a start: 200 | without a terminal: 2248
sessions with a sequence gap (max_seq != rows): 700 | with repeated sequence numbers: 0
sessions mixing tenants: 727 | mixing release_version spellings: 1888


In [49]:
st = d[d.event_type == "session.start"].set_index("session_id").event_ts
term = d[d.event_type.isin(["session.end", "session.error"])].set_index("session_id")
dur = (term.event_ts - st.reindex(term.index)).dt.total_seconds()
cross = (term.latency_ms.astype("float") / 1000 - dur).dropna()
print("derived duration (s):"); print(dur.describe(percentiles=[.5, .95]).round(2).to_string())
print(f"terminal latency_ms disagrees with derived duration by > 1 s in {int((cross.abs() > 1).sum())} "
      f"of {len(cross)} sessions")

derived duration (s):
count    98285.00
mean        19.65
std         10.20
min          2.34
50%         17.61
95%         39.53
max        109.49
terminal latency_ms disagrees with derived duration by > 1 s in 1942 of 97952 sessions


In [50]:
F.add("AC15", "agent_activity.parquet", "sessions with no session.start (single orphan rows)",
      "count of session.start per session", int((per.starts == 0).sum()), "reject ORPHAN_EVENT")
F.add("AC16", "agent_activity.parquet", "sessions without a terminal event / with sequence gaps",
      "terminal count and max(sequence_number) vs row count per session",
      f"{int((per.terminals == 0).sum())} / {int((per.max_seq != per.rows).sum())}",
      "not rejected; excluded from duration, counted in trace completeness")
F.add("AC17", "agent_activity.parquet", "terminal latency_ms disagrees with derived duration",
      "abs(latency_ms/1000 - (terminal - start)) > 1 s", int((cross.abs() > 1).sum()),
      "use derived duration only; report disagreement count")

### 7.10 Agent versions vs `launch_date` (basis for the Type-2 validity windows)

In [51]:
vr = d.assign(v=d.release_version.str.lstrip("v")).groupby(["agent_id", "v"]).event_ts.agg(["min", "max"])
vr = vr.join(agents.assign(v=agents.release_version.str.lstrip("v")).drop_duplicates(["agent_id", "v"])
             .set_index(["agent_id", "v"])[["launch_date", "is_deprecated"]])
vr

min                              max launch_date is_deprecated
agent_id v                                                                                                
ag_001   1.0.0 2026-01-31 00:00:33.753000+00:00 2026-03-31 23:37:42.472000+00:00  2025-10-01         false
ag_002   2.1.0 2026-01-31 00:29:50.876000+00:00 2026-02-24 23:36:00.954000+00:00  2025-11-01          true
         2.2.0 2026-02-25 00:20:55.281000+00:00 2026-03-31 23:48:57.216000+00:00  2026-02-25         false
ag_003   0.5.0 2026-01-31 00:52:14.173000+00:00 2026-03-04 23:59:43.578000+00:00  2025-12-01          true
         0.6.0 2026-03-05 00:11:48.673000+00:00 2026-03-31 22:57:20.473000+00:00  2026-03-05         false
ag_004   3.0.1 2026-01-31 00:00:48.906000+00:00 2026-02-19 23:57:01.396000+00:00  2025-08-14          true
         3.0.2 2026-02-20 00:04:17.540000+00:00 2026-03-31 23:54:31.573000+00:00  2026-02-20         false
ag_007   2.2.0 2026-01-31 00:10:33.943000+00:00 2026-02-09 23:24:38.021000+00:00  2025-12-01          true
         2.3.1 2026-02-10 00:10:05.072000+00:00 2026-03-31 22:02:02.947000+00:00  2026-02-10         false
ag_008   1.0.0 2026-01-31 00:05:55.131000+00:00 2026-03-31 20:15:41.133000+00:00  2025-11-20         false
ag_011   1.4.0 2026-01-31 00:28:23.596000+00:00 2026-03-31 23:47:57.631000+00:00  2026-01-20         false
ag_012   3.1.0 2026-01-31 00:40:18.467000+00:00 2026-03-07 23:44:35.313000+00:00  2025-10-10          true
         3.2.0 2026-03-08 00:25:48.492000+00:00 2026-03-31 23:28:39.692000+00:00  2026-03-08         false
ag_019   0.9.0 2026-01-31 01:02:07.595000+00:00 2026-03-31 23:19:57.769000+00:00  2026-01-10         false
ag_020   1.0.0 2026-01-31 00:10:18.982000+00:00 2026-03-09 22:37:50.764000+00:00  2025-09-01          true
         1.1.0 2026-03-10 01:47:39.017000+00:00 2026-03-31 23:41:08.997000+00:00  2026-03-10         false
ag_021   1.2.0 2026-01-31 00:49:16.944000+00:00 2026-03-31 22:20:29.643000+00:00  2025-12-15         false
ag_030   1.0.0 2026-01-31 00:25:45.743000+00:00 2026-02-28 23:52:44.941000+00:00  2025-12-20          true
         1.1.0 2026-03-01 00:41:13.297000+00:00 2026-03-31 23:21:10.334000+00:00  2026-03-01         false
ag_031   0.3.0 2026-01-31 00:21:20.550000+00:00 2026-03-31 23:49:51.850000+00:00  2026-01-05         false
ag_040   1.0.0 2026-01-31 00:24:50.358000+00:00 2026-02-24 23:45:02.464000+00:00  2025-09-30         false
ag_321   9.9.9 2026-01-31 09:32:59.190000+00:00 2026-03-31 17:24:33.069000+00:00         NaN           NaN
ag_999   9.9.9 2026-01-31 08:45:25.910000+00:00 2026-03-31 21:38:36.515000+00:00         NaN           NaN

### 7.11 Where traffic concentrates (input to the Spark skew work in M2)

In [52]:
conc = (d.groupby(["tenant_id", "agent_id"]).size().rename("rows").reset_index()
        .assign(share=lambda x: (x.rows / x.rows.sum()).round(4)).sort_values("rows", ascending=False))
conc.head(8)

,tenant_id,agent_id,rows,share
3,tn_001,ag_004,440005,0.4731
0,tn_001,ag_001,55443,0.0596
4,tn_002,ag_019,49958,0.0537
13,tn_005,ag_030,49656,0.0534
14,tn_005,ag_031,46306,0.0498
7,tn_003,ag_007,45168,0.0486
1,tn_001,ag_002,41131,0.0442
5,tn_002,ag_020,39523,0.0425


## 8. `network_flows.csv`

In [53]:
profile_frame(flows)

,column,dtype,nulls,null_rate,empty_str,distinct,top_values
0,flow_id,str,0,0.0000,0,42392,"nf_000001 (1), nf_000002 (1), nf_000003 (1)"
1,session_id,str,0,0.0000,0,27858,"ss_069182 (7), ss_061656 (6), ss_045479 (6)"
2,sandbox_instance_id,str,0,0.0000,0,28,"sbx_01fa (2820), sbx_01ed (2692), sbx_0207 (2631)"
3,flow_time,str,0,0.0000,0,42308,"2026-01-31T19:29:00.444Z (2), 2026-02-01T14:56:18.252Z (2), 2026-02-02T18:40:50.066Z (2)"
4,src_ip,str,0,0.0000,0,28,"10.20.12.11 (2820), 10.20.12.10 (2692), 10.20.12.12 (2631)"
5,dst_host,str,3514,0.0829,0,11,"mail.kestrel.ai (11342), research.marketwire.example (6816), stats.public-data.example..."
6,dst_ip,str,0,0.0000,0,63,"198.51.100.11 (11672), 198.51.100.22 (7012), 198.51.100.31 (6157)"
7,dst_port,str,0,0.0000,0,4,"443 (40304), 3389 (720), 22 (709)"
8,protocol,str,0,0.0000,0,4,"https (40304), rdp (720), ssh (709)"
9,bytes_sent,str,0,0.0000,0,7362,"512 (733), 128 (693), 256 (684)"


In [54]:
print("flow_time shapes:", value_shapes(flows.flow_time).to_dict())
print("protocol outside enum:", dict(enum_violations(flows.protocol, ["https", "http", "ssh", "smb", "dns", "rdp"])))
print("verdict outside enum:", dict(enum_violations(flows.verdict, ["allowed", "blocked", "logged_only"])))
print("duplicate flow_id:", int(flows.flow_id.duplicated().sum()))
pd.crosstab(flows.protocol, flows.verdict)

flow_time shapes: {'9999-99-99A99:99:99.999A': 42392}
protocol outside enum: {}
verdict outside enum: {}
duplicate flow_id: 0


verdict,allowed,blocked,logged_only
protocol,,,
https,38644,1233,427
rdp,0,720,0
smb,0,659,0
ssh,0,709,0


In [55]:
print("empty dst_host by protocol:", flows[flows.dst_host.isna()].protocol.value_counts().to_dict())
named = flows.dropna(subset=["dst_host"])
ip_hosts = named.groupby("dst_ip").dst_host.nunique()
print("dst_ip with more than one host name:", int((ip_hosts > 1).sum()))
fillable = flows.dst_host.isna() & flows.dst_ip.isin(ip_hosts.index)
print("empty dst_host whose dst_ip has a known name elsewhere:", int(fillable.sum()))
print("geo_country null:", int(flows.geo_country.isna().sum()))
numeric_ranges(flows, ["dst_port", "bytes_sent", "bytes_received"])

empty dst_host by protocol: {'https': 1426, 'rdp': 720, 'ssh': 709, 'smb': 659}
dst_ip with more than one host name: 0
empty dst_host whose dst_ip has a known name elsewhere: 1192
geo_country null: 2088


,column,count,min,p50,p99,max,negative,zero
0,dst_port,42392,22,443.0,3389.00,3389,0,0
1,bytes_sent,42392,128,1373.0,30414.09,65525,0,0
2,bytes_received,42392,0,90218.0,197983.00,199999,0,3321


In [56]:
sessions = set(act.session_id)
orph = flows[~flows.session_id.isin(sessions)]
print("flows whose session_id is not in activity:", len(orph))
sbx_of_session = d.drop_duplicates("session_id").set_index("session_id").sandbox_instance_id
print("flows whose sandbox_instance_id differs from the session's:",
      int((flows.sandbox_instance_id != flows.session_id.map(sbx_of_session)).sum()))
print("src_ip per sandbox (max):", flows.groupby("sandbox_instance_id").src_ip.nunique().max(),
      "| sandboxes per src_ip (max):", flows.groupby("src_ip").sandbox_instance_id.nunique().max())

flows whose session_id is not in activity: 84
flows whose sandbox_instance_id differs from the session's: 84
src_ip per sandbox (max): 1 | sandboxes per src_ip (max): 1


In [57]:
# clock check: a flow should follow the tool call that caused it (same session, target = host)
flow_ts = pd.to_datetime(flows.flow_time, utc=True, format="ISO8601")
calls = d[d.event_type == "tool.call"][["session_id", "target_resource", "event_ts"]]
m = flows.assign(flow_ts=flow_ts).merge(calls, left_on=["session_id", "dst_host"], right_on=["session_id", "target_resource"])
m["delta_s"] = (m.flow_ts - m.event_ts).dt.total_seconds()
m = m.dropna(subset=["delta_s"])          # calls with an unparseable event_time cannot be paired
best = m.loc[m.delta_s.abs().groupby(m.flow_id).idxmin()]
print(f"{best.flow_id.nunique():,} flows paired with a tool call; flow_time - call start (s):")
print(best.delta_s.describe(percentiles=[.01, .5, .99]).round(3).to_string())

37,602 flows paired with a tool call; flow_time - call start (s):
count    37602.000
mean         0.339
std          0.683
min        -25.681
1%           0.099
50%          0.335
99%          0.572
max         29.111


In [58]:
F.add("NF1", "network_flows.csv", "flows referencing a session_id absent from activity",
      "FK check vs activity session_id", len(orph), "reject ORPHAN_EVENT")
F.add("NF2", "network_flows.csv", "dst_host empty", "null check, by protocol", int(flows.dst_host.isna().sum()),
      "fill from dst_ip->host map where known (flag DST_HOST_FILLED), else keep dst_ip as target")
F.add("NF3", "network_flows.csv", "geo_country empty", "null check", int(flows.geo_country.isna().sum()),
      "keep null (documented nullable)")
F.add("NF4", "network_flows.csv", "flow_time is genuine UTC",
      "median flow_time - tool.call start = +0.34 s, no hour offset", int(best.flow_id.nunique()),
      "use as-is; flows are the clock anchor for the audit-log offset")

## 9. `sandbox_audit.log` (syslog text, regex-parsed)

Lines are matched against an anchored regex in `readers.AUDIT_LINE_RE`; anything that does not match end-to-end is
kept with its raw text and line number.

In [59]:
un = audit_x.unparsed
print(f"parsed {len(audit):,} lines, unparsed {len(un):,}:", un.problem.value_counts().to_dict())
lens = un.raw_line.str.len()
print("unparsed line length: min", lens.min(), "max", lens.max(), "vs parsed median", int(audit.raw_line.str.len().median()))
un.head(5)

parsed 183,481 lines, unparsed 524: {'regex_no_match': 524}
unparsed line length: min 60 max 171 vs parsed median 185


,source_file,source_line,raw_line,problem
0,sandbox_audit.log,129,Jan 31 03:46:27 sbx-eu-102 kestrel-sandbox[2022]: session=ss_09,regex_no_match
1,sandbox_audit.log,285,Jan 31 06:35:47 sbx-eu-100 kestrel-sandbox[2000]: session=ss_054184 instance=sbx_01a0 ...,regex_no_match
2,sandbox_audit.log,358,Jan 31 07:46:07 sbx-eu-102 kestrel-sandbox[2022]: session=ss_015763 instance=sbx_01ba ...,regex_no_match
3,sandbox_audit.log,626,Jan 31 10:12:56 sbx-eu-133 kestrel-sandbox[3067]: session=ss_049581 instance=sbx_01ed ...,regex_no_match
4,sandbox_audit.log,1708,Jan 31 16:14:09 sbx-me-129 kestrel-sandbox[2895]: session=ss_040738 instance=sbx_01f9 ...,regex_no_match


In [60]:
profile_frame(audit.drop(columns=['raw_line']))

,column,dtype,nulls,null_rate,empty_str,distinct,top_values
0,log_time,str,0,0.0,0,177921,"Feb 9 11:15:09 (5), Feb 5 19:29:43 (4), Feb 7 14:57:59 (4)"
1,host,str,0,0.0,0,42,"sbx-eu-100 (38543), sbx-eu-102 (37922), sbx-eu-101 (37860)"
2,process,str,0,0.0,0,1,kestrel-sandbox (183481)
3,pid,str,0,0.0,0,42,"2000 (38543), 2022 (37922), 2011 (37860)"
4,session_id,str,0,0.0,0,74985,"ss_079460 (10), ss_061656 (9), ss_092107 (9)"
5,sandbox_instance_id,str,0,0.0,0,42,"sbx_01a0 (38543), sbx_01ba (37922), sbx_01ad (37860)"
6,decision,str,0,0.0,0,2,"ALLOW (177018), DENY (6463)"
7,action,str,0,0.0,0,6,"process_spawn (91171), file_write (48195), socket_connect (42179)"
8,target,str,0,0.0,0,573,"/usr/bin/systemctl (28262), /usr/bin/journalctl (28148), /usr/bin/df (28033)"
9,reason,str,0,0.0,0,10,"TOOL_ALLOWED (89939), PATH_IN_SCOPE (48195), EGRESS_ALLOWED (34754)"


In [61]:
print("'-' (null marker) per field:", {c: int((audit[c] == "-").sum()) for c in audit.columns if (audit[c] == "-").any()})
print("decision:", audit.decision.value_counts().to_dict())
print("action outside enum:", dict(enum_violations(audit.action, ["socket_connect", "file_read", "file_write",
                                                               "process_spawn", "setuid", "mount", "ptrace"])))
print("months in log_time:", audit.log_time.str[:3].value_counts().to_dict())
print("exact duplicate lines:", int(audit.raw_line.duplicated().sum()))
print("session_id not in activity:", int((~audit.session_id.isin(sessions)).sum()))
print("instances per host (max):", audit.groupby("host").sandbox_instance_id.nunique().max(),
      "| hosts per instance (max):", audit.groupby("sandbox_instance_id").host.nunique().max())
pd.crosstab(audit.reason, audit.decision)

'-' (null marker) per field: {'policy_version': 2446}
decision: {'ALLOW': 177018, 'DENY': 6463}
action outside enum: {}
months in log_time: {'Mar': 96535, 'Feb': 84098, 'Jan': 2744, 'Apr': 104}
exact duplicate lines: 4
session_id not in activity: 0
instances per host (max): 1 | hosts per instance (max): 1


decision,ALLOW,DENY
reason,,
EGRESS_ALLOWED,34754,0
EGRESS_NOT_ALLOWED,0,1475
ESCALATION_FORBIDDEN,0,877
NO_NETWORK_POLICY,3705,0
PATH_IN_SCOPE,48195,0
PATH_OUT_OF_SCOPE,0,1059
PORT_DENIED,0,1820
TOOL_ALLOWED,89939,0
TOOL_NOT_ALLOWED,0,1232


### 9.1 Establishing the hosts' UTC offset

The log has no offset and no year. Year: every month is Jan–Apr and the pack covers 2026-01-31 → 2026-03-31, so
the year is 2026; the 104 `Apr` lines already hint at a positive offset (local clock past midnight on 1 April).
Offset: every `socket_connect` decision should coincide with the network flow it allowed or blocked, and flows are
proven UTC (section 8). Pair them on `(session_id, target = dst_host:port or dst_ip:port)` and read the offset off the
time difference, per host and per month (a DST switch would show up as a 1 h jump in late March).

In [62]:
a = audit.assign(local=pd.to_datetime("2026 " + audit.log_time, format="%Y %b %d %H:%M:%S"))
fl = flows.assign(flow_naive=flow_ts.dt.tz_localize(None),
                  t_host=flows.dst_host + ":" + flows.dst_port, t_ip=flows.dst_ip + ":" + flows.dst_port)
sc_ = a[a.action == "socket_connect"]
pairs = pd.concat([sc_.merge(fl, left_on=["session_id", "target"], right_on=["session_id", "t_host"]),
                   sc_.merge(fl, left_on=["session_id", "target"], right_on=["session_id", "t_ip"])])
pairs["offset_h"] = (pairs.local - pairs.flow_naive).dt.total_seconds() / 3600
by_host = pairs.groupby("host").offset_h.agg(pairs="size", median="median", min="min", max="max").round(4)
by_host["drift_s_vs_2h"] = ((by_host["median"] - 2) * 3600).round(0)
by_host

,pairs,median,min,max,drift_s_vs_2h
host,,,,,
sbx-eu-100,1536,1.9858,1.9760,1.9954,-51.0
sbx-eu-101,1353,2.0114,1.9990,2.0238,41.0
sbx-eu-102,1357,1.9768,1.9664,1.9869,-84.0
sbx-eu-106,1769,1.9870,1.9737,2.0003,-47.0
sbx-eu-107,1639,1.9966,1.9822,2.0109,-12.0
sbx-eu-108,1771,2.0093,1.9998,2.0186,33.0
sbx-eu-109,2273,1.9940,1.9802,2.0079,-22.0
sbx-eu-110,2190,2.0011,1.9822,2.0201,4.0
sbx-eu-111,2376,1.9890,1.9777,2.0005,-40.0


In [63]:
pairs["region"] = pairs.host.str.split("-").str[1]
pairs["week"] = pairs.flow_naive.dt.to_period("W").astype(str)
pairs.pivot_table(index="week", columns="region", values="offset_h", aggfunc="median").round(3)

region,eu,me,us
week,,,
2026-01-26/2026-02-01,2.009,2.007,1.992
2026-02-02/2026-02-08,2.009,2.006,2.006
2026-02-09/2026-02-15,2.009,2.007,1.992
2026-02-16/2026-02-22,2.009,2.007,1.992
2026-02-23/2026-03-01,2.006,2.007,1.991
2026-03-02/2026-03-08,2.001,2.006,1.994
2026-03-09/2026-03-15,2.008,2.007,1.998
2026-03-16/2026-03-22,2.001,2.007,1.991
2026-03-23/2026-03-29,2.007,2.007,1.991


Flows only cover hosts whose sandboxes connect to the network (27 of 42). Every decision, though, has a causing
**tool call** (`process_spawn` → shell command, `file_write` → path, …), so pairing on `(session_id, target =
target_resource)` measures the offset for **all** hosts, and the 27 shared hosts give an independent cross-check.

In [64]:
from kestrel_m1 import timeutil as tu
tool_calls = d[d.event_type == "tool.call"]
by_calls = tu.audit_offsets_from_tool_calls(audit, tool_calls).set_index("host")
by_flows = tu.audit_offsets_from_flows(audit, flows).set_index("host")
both = by_calls[["pairs", "skew_vs_utc_plus_2_s"]].join(by_flows[["pairs", "skew_vs_utc_plus_2_s"]], rsuffix="_flows")
both["anchors_disagree_s"] = (both.skew_vs_utc_plus_2_s - both.skew_vs_utc_plus_2_s_flows).round(2)
print(f"hosts measured: {len(both)} of {audit.host.nunique()} | cross-checked with flows: {both.anchors_disagree_s.notna().sum()} "
      f"| max disagreement between anchors: {both.anchors_disagree_s.abs().max()} s")
both.round(1).sort_values("skew_vs_utc_plus_2_s")

hosts measured: 42 of 42 | cross-checked with flows: 27 | max disagreement between anchors: 0.37 s


,pairs,skew_vs_utc_plus_2_s,pairs_flows,skew_vs_utc_plus_2_s_flows,anchors_disagree_s
host,,,,,
sbx-me-128,366,-90.3,NaN,NaN,NaN
sbx-us-114,833,-90.2,NaN,NaN,NaN
sbx-us-118,170,-90.0,1721.0,-90.1,0.1
sbx-eu-102,36688,-83.4,1176.0,-83.5,0.1
sbx-me-126,103,-82.8,1153.0,-82.9,0.1
sbx-me-129,379,-79.3,NaN,NaN,NaN
sbx-me-130,283,-76.5,NaN,NaN,NaN
sbx-eu-138,3882,-65.8,NaN,NaN,NaN
sbx-us-112,831,-56.5,NaN,NaN,NaN


In [65]:
fixed = tu.audit_local_to_utc(audit.log_time)
corrected = tu.audit_local_to_utc(audit.log_time, audit.host, by_calls.median_offset_s.to_dict())
fl_t = flows.assign(ft=flow_ts, t=flows.dst_host + ":" + flows.dst_port)
for name, u in [("fixed UTC+2", fixed), ("UTC+2 + per-host skew", corrected)]:
    mm = audit.assign(u=u)[audit.action == "socket_connect"].merge(fl_t, left_on=["session_id", "target"],
                                                                     right_on=["session_id", "t"])
    dd = (mm.ft - mm.u).dt.total_seconds()
    print(f"{name:24s} flow - audit decision: p05={dd.quantile(.05):7.2f}s  p50={dd.median():6.2f}s  p95={dd.quantile(.95):6.2f}s"
          f" | earliest decision {u.min()}")

fixed UTC+2              flow - audit decision: p05= -80.22s  p50=-23.11s  p95= 83.87s | earliest decision 2026-01-30 23:59:28+00:00
UTC+2 + per-host skew    flow - audit decision: p05=  -6.97s  p50=  0.15s  p95=  7.28s | earliest decision 2026-01-31 00:00:51.443000+00:00


In [66]:
# overlap PO1 evidence: which version did the sandbox itself apply to research sessions inside the overlap window?
prof = agents.assign(v=agents.release_version.str.lstrip("v")).drop_duplicates(["agent_id", "v"])
sess_profile = (d[d.event_type == "session.start"].assign(v=lambda x: x.release_version.str.lstrip("v"))
                .merge(prof[["agent_id", "v", "sandbox_profile_id"]], on=["agent_id", "v"])
                .set_index("session_id").sandbox_profile_id)
a["utc_approx"] = a.local - pd.Timedelta(hours=2)
win = a[(a.session_id.map(sess_profile) == "sp_research_std") & (a.utc_approx < "2026-02-01")]
print("research-profile audit lines inside the v2/v3 overlap, by policy_version:", win.policy_version.value_counts().to_dict())
F.add("PO1b", "sandbox_policies.json", "which version applies inside the sp_research_std v2/v3 overlap",
      "audit-log policy_version for research sessions between 2026-01-25 and 2026-02-01 (UTC)", len(win),
      f"the sandbox itself applied {sorted(win.policy_version.unique())}: newer version wins inside an overlap")

research-profile audit lines inside the v2/v3 overlap, by policy_version: {'3': 346}


In [67]:
F.add("AU1", "sandbox_audit.log", "truncated lines that do not match the syslog regex",
      "anchored regex; length of failing lines", len(un), "reject UNPARSEABLE_RECORD with raw line + line number")
F.add("AU2", "sandbox_audit.log", "policy_version is '-' (null marker)", "count of '-' per field",
      int((audit.policy_version == "-").sum()), "null; derive policy_version_applied by validity-window join, flag")
F.add("AU3", "sandbox_audit.log", "exact duplicate lines", "raw_line.duplicated()",
      int(audit.raw_line.duplicated().sum()), "keep first, later copies rejected DUPLICATE_EVENT_ID")
F.add("AU4", "sandbox_audit.log", "log_time is host-local with no year and no offset; all hosts are UTC+2, "
      "regardless of the eu/us/me hostname", "socket_connect paired with UTC flows: per-host median offset 1.97-2.03 h, "
      "no DST jump; tool-call pairing covers all 42 hosts and agrees with flows within 0.4 s", int(len(pairs)),
      "utc = local(2026) - measured per-host offset (UTC+2 + skew up to ~90 s); fallback -2 h for unseen hosts")

## 10. `siem_alerts.json` (envelope + nested alerts)

In [68]:
print("envelope:", siem_x.envelope)
profile_frame(siem)

envelope: {'generated_at': '2026-04-01T02:00:00+02:00', 'source': 'corp-siem-export-v2'}


,column,dtype,nulls,null_rate,empty_str,distinct,top_values
0,alert_id,str,0,0.0,0,3754,"al_00001 (1), al_00002 (1), al_00003 (1)"
1,source_system,str,0,0.0,0,3,"corp-siem (1912), cloud-ids (1251), egress-proxy (591)"
2,severity,str,0,0.0,0,4,"high (1573), medium (962), low (728)"
3,detected_at,str,0,0.0,0,3730,"2026-01-31T18:39:56+02:00 (2), 2026-02-07T14:01:58+02:00 (2), 2026-02-14T10:18:10+02:0..."
4,source_ip,str,0,0.0,0,975,"10.20.8.10 (237), 10.20.8.12 (221), 10.20.8.11 (209)"
5,rule,str,0,0.0,0,9,"rdp_lateral_attempt (548), internal_ssh_attempt (526), lateral_movement_smb_burst (499)"
6,confidence,float64,0,0.0,0,76,"0.49 (81), 0.62 (79), 0.57 (75)"
7,destination_host,str,0,0.0,0,56,"203.0.113.45 (168), 203.0.113.47 (166), 203.0.113.46 (157)"
8,destination_port,int64,0,0.0,0,4,"22 (1074), 3389 (1073), 445 (1016)"
9,technique_id,str,0,0.0,0,6,"T1021.001 (962), T1021.004 (957), T1021.002 (905)"


In [69]:
print("severity outside enum:", dict(enum_violations(siem.severity, ["low", "medium", "high", "critical"])))
print("detected_at shapes:", value_shapes(siem.detected_at).to_dict())
print("destination_host is an IPv4 address:", bool(siem.destination_host.str.fullmatch(r"\d+\.\d+\.\d+\.\d+").all()))
print("duplicate alert_id:", int(siem.alert_id.duplicated().sum()))
siem.assign(offset=siem.detected_at.str[-6:], src_ip_in_flows=siem.source_ip.isin(set(flows.src_ip))) \
    .pivot_table(index="source_system", columns="offset", values="alert_id", aggfunc="count", fill_value=0)

severity outside enum: {}
detected_at shapes: {'9999-99-99A99:99:99+99:99': 2503, '9999-99-99A99:99:99-99:99': 1251}
destination_host is an IPv4 address: True
duplicate alert_id: 0


offset,+02:00,+03:00,-05:00
source_system,,,
cloud-ids,0,0,1251
corp-siem,1912,0,0
egress-proxy,0,591,0


In [70]:
s = siem.assign(t=pd.to_datetime(siem.detected_at, utc=True, format="ISO8601"),
                src_ip_in_flows=siem.source_ip.isin(set(flows.src_ip)))
print(pd.crosstab(s.rule, s.src_ip_in_flows).to_string())

src_ip_in_flows                      False  True 
rule                                             
internal_port_sweep                      0    339
internal_ssh_attempt                     0    526
internal_ssh_attempt_external          431      0
lateral_movement_smb_burst               0    499
lateral_movement_smb_burst_external    406      0
rdp_lateral_attempt                      0    548
rdp_lateral_attempt_external           414      0
unclassified_destination                 0    100
unclassified_destination_upload          0    491


In [71]:
cand = s.merge(flows.assign(flow_ts=flow_ts, dst_port_i=flows.dst_port.astype(int)),
               left_on=["source_ip", "destination_host", "destination_port"],
               right_on=["src_ip", "dst_ip", "dst_port_i"])
cand["delta_s"] = (cand.t - cand.flow_ts).dt.total_seconds()
nearest = cand.loc[cand.delta_s.abs().groupby(cand.alert_id).idxmin()]
print("alert - nearest flow (s), by source_system (after UTC conversion):")
print(nearest.groupby("source_system").delta_s.describe(percentiles=[.05, .5, .95]).round(1).to_string())
rows = []
for N in [5, 10, 30, 60, 120, 300, 600]:
    c = cand[cand.delta_s.abs() <= N].groupby("alert_id").size()
    rows.append({"N_s": N, "matched": int((c == 1).sum()), "ambiguous": int((c > 1).sum()), "unmatched": len(s) - len(c)})
pd.DataFrame(rows)

alert - nearest flow (s), by source_system (after UTC conversion):
                count  mean   std  min   5%   50%   95%    max
source_system                                                 
corp-siem      1912.0  27.4  40.3 -1.6  2.3  21.8  43.1  390.2
egress-proxy    591.0  31.3  54.3  0.4  2.9  21.6  43.6  393.4


,N_s,matched,ambiguous,unmatched
0,5,285,2,3467
1,10,579,4,3171
2,30,1654,16,2084
3,60,2409,27,1318
4,120,2409,27,1318
5,300,2453,27,1274
6,600,2475,28,1251


In [72]:
F.add("SI1", "siem_alerts.json", "detected_at uses a different UTC offset per source_system (+02:00, +03:00, -05:00)",
      "offset suffix x source_system crosstab; deltas to flows ~0-45 s after UTC conversion (offsets are honest)",
      len(siem), "convert with the stated offset to UTC")
F.add("SI2", "siem_alerts.json", "cloud-ids '*_external' alerts come from source IPs that never appear in flows",
      "source_ip membership in flows.src_ip, by rule", int((~s.src_ip_in_flows).sum()),
      "leave unmatched (activity outside the platform); never forced")
F.add("SI3", "siem_alerts.json", "no session_id; correlation needs a tolerance N",
      "N-sensitivity table: counts plateau from N = 60 s", len(siem),
      "rule src_ip=source_ip, dst_ip=destination.host, port, |dt| <= 60 s; report matched/ambiguous/unmatched")

### Established time conventions (evidence above)

| Source | Field | Convention found | Evidence |
|---|---|---|---|
| agent_activity | `ingest_time` | ISO-8601 UTC with `Z`, one format | shapes (7.5) |
| agent_activity | `event_time` | UTC in every parseable format; epoch is **milliseconds**; slash dates are **day-first** | lateness ≈ 1 s per format, day field > 12 (7.5) |
| network_flows | `flow_time` | genuine UTC | +0.34 s after the causing tool call (8) |
| sandbox_audit.log | `log_time` | host-local **UTC+2**, no DST, all hosts incl. `us`/`me`; year 2026; per-host clock skew −90…+87 s, corrected | socket_connect vs flows and decisions vs tool calls, per host/week (9.1) |
| siem_alerts | `detected_at` | offset stated per record and honest (+02:00 corp-siem, +03:00 egress-proxy, −05:00 cloud-ids) | alerts land 0–45 s after their flow once converted (10) |
| sandbox_policies / pricing / tenants | validity & audit dates | ISO-8601 UTC `Z` | shapes |

## 11. Findings log

Every problem above, how it was detected, how many records it touches, and the decision planned for the cleaning
module. Exported for the data-quality findings document.

In [73]:
findings = F.frame()
out_dir = ROOT / "output" / "profiling"
out_dir.mkdir(parents=True, exist_ok=True)
findings.to_csv(out_dir / "findings.csv", index=False)
print(len(findings), "findings ->", out_dir / "findings.csv")
findings

45 findings -> d:\DE\Tata-engineering_agents\M1\output\profiling\findings.csv


,id,source,problem,detected_by,count,planned_decision
0,T1,tenants.csv,tn_003 'NILEBANK Ltd.' and tn_004 'nilebank' are the same company,normalised name + identical security_contact/onboarded_at,2,entity-resolve tn_004 -> tn_003 (tn_004 has no activity)
1,T2,tenants.csv,region 'eu-central-9' is not in the region enum,enum check,1,"set region null, quality flag INVALID_ENUM_REGION"
2,T3,tenants.csv,security_contact missing,null check,1,keep null (nullable attribute)
3,T4,tenants.csv,inactive tenant tn_006 still has historical traffic,join to activity,15043,keep: documented as legitimate history
4,AG1,agents.csv,ag_007 'v2.3.1' duplicates 2.3.1 with a different agent_name and autonomy_level,"strip 'v' prefix, then duplicate natural key",1,"drop the v-prefixed row, keep canonical 2.3.1 attributes; record in entity_resolution"
5,AG2,agents.csv,ag_019 points to sandbox profile 'sp_missing_profile' that does not exist,FK check against sandbox_policies,1,keep agent; policy_version_applied null and quality flag NO_POLICY_PROFILE for its rec...
6,TL1,tools.csv,tl_vector_search 2.0.1 appears twice (identical rows),duplicated() on all columns,1,"deduplicate on (tool_id, tool_version)"
7,TL2,tools.csv,tl_web_fetch is named web_fetch / http_get / webfetch across versions,nunique(tool_name) per tool_id,3,build alias map tool_name -> tool_id for entity resolution
8,TL3,tools.csv,tl_code_exec category is code_exec in 0.9.1 but shell in 0.9.0,nunique(tool_category) per tool_id,1,"keep per-version category, quality flag; document"
9,P1,model_pricing.csv,mdl_kst_nano_1 has empty input/output prices,null check,1,"cost_usd null for nano turns, quality flag PRICE_MISSING; declared in the cost policy"
